# The InConvenience of Building RAG Without Langchain

1. Read the document
2. Split the document
    - May fail to generate a response due to token limit
    - Longer documents (longer input) take more time to generate responses
3. Embedding: Store in a vector database
4. When a question is asked, perform similarity search in the vector database
5. Pass the retrieved documents along with the question to the LLM

# 1. Read the document (문서의 내용을 읽는다)

In [ ]:
# langchain을 썼을 때는 document loader를 썼는데, 지금은 langchain을 안쓰니까 -> python-docx 이용
# %pip install python-docx

- docx is a ZIP-compressed XML file, so `open().read()` returns broken binary
- `python-docx` parses the internal XML and extracts text via `document.paragraphs`
- `paragraph.text` contains the full text of each paragraph (split by line breaks)
- All paragraphs are concatenated into `full_text` for later chunking

docx file -> parse with Document -> paragraph별 text 추출 -> full_text(usable)

In [ ]:
from docx import Document

document = Document("./tax.docx")
print(f"document == {document}")
# dir(document) : document가 가지고 있는 모든 속성과 method 이름 list 보여줌
print(f"document == {dir(document)}")

full_text = ""
for index, paragraph in enumerate(document.paragraphs):
    print(f"paragraph == {paragraph.text}")
    full_text += f"{paragraph.text}\n"

# 2. Split text (문서를 쪼갠다)

In [ ]:
# %pip install tiktoken

In [ ]:
# tiktoken : split text
import tiktoken


def split_text(full_text, chunk_size):
    encoder = tiktoken.encoding_for_model("gpt-4o")
    # encode = text -> 숫자 list
    total_encoding = encoder.encode(full_text)
    total_token_count = len(total_encoding)
    text_list = []

    for i in range(0, total_token_count, chunk_size):
        chunk = total_encoding[i : i + chunk_size]
        decoded = encoder.decode(chunk)
        text_list.append(decoded)

    return text_list

In [ ]:
chunk_list = split_text(full_text, 1500)

In [ ]:
chunk_list

# 3. Document Embedding

In [ ]:
# chroma가 제공하는 기본 기능이 있다. 그걸 이용해 embedding

# %pip install chromadb

In [ ]:
import chromadb

chroma_client = chromadb.Client()

In [ ]:
collection_name = "tax-collection"

In [ ]:
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction

ollama_embedding = OllamaEmbeddingFunction(model_name="nomic-embed-text")

In [ ]:
tax_collection = chroma_client.get_or_create_collection(
    collection_name, embedding_function=ollama_embedding
)

In [ ]:
id_list = []
for index in range(len(chunk_list)):
    id_list.append(f"{index}")

In [ ]:
len(id_list)

In [ ]:
len(chunk_list)

In [ ]:
tax_collection.add(documents=chunk_list, ids=id_list)

# 4. Similarity Check (유사도 검색)

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_doc = tax_collection.query(query_texts=query)

In [ ]:
retrieved_doc["documents"][0]

# 5. LLM 질의

In [ ]:
import ollama

response = ollama.chat(
    model="llama3:latest",
    messages=[
        {
            "role": "system",
            "content": f'당신은 한국의 소득세 전문가 입니다. 아래 내용을 참고해서 사용자의 질문에 답변해주세요 {retrieved_doc["documents"][0]}',
        },
        {"role": "user", "content": query},
    ],
)

In [ ]:
response

In [ ]:
response.message.content

# LangChain RAG vs 순수 Python RAG 비교

RAG의 5단계를 기준으로 비교합니다.

---

## 1. 문서 읽기 (Read Document)

| LangChain | 순수 Python |
|-----------|-------------|
| `Docx2txtLoader("./tax.docx")` 한 줄 | `python-docx`로 Document 객체 만들고, paragraphs를 for문 돌면서 직접 `full_text`에 합침 |

LangChain은 다양한 파일 포맷(docx, pdf, csv 등)에 대해 통일된 `loader.load()` 인터페이스를 제공하지만, 직접 하면 파일 포맷마다 별도 라이브러리(`python-docx`, `PyPDF2` 등)를 써서 직접 파싱해야 합니다.

---

## 2. 문서 분할 (Split Text)

| LangChain | 순수 Python |
|-----------|-------------|
| `RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)` + `load_and_split()` | `tiktoken`으로 직접 encode → 슬라이싱 → decode하는 함수를 직접 작성 |

LangChain은 `chunk_overlap`(청크 간 겹침)을 기본 지원해서 문맥이 잘리는 걸 방지합니다. 직접 구현한 버전은 overlap 없이 단순히 토큰을 잘라서, 문장 중간에서 잘릴 수 있습니다.

---

## 3. Embedding & 벡터DB 저장

| LangChain | 순수 Python |
|-----------|-------------|
| `OllamaEmbeddings()` + `Chroma.from_documents()`로 한번에 임베딩+저장 | `chromadb.Client()` 직접 생성, collection 만들고, id 리스트 직접 만들어서 `collection.add()` |

LangChain은 Document 객체를 그대로 넘기면 알아서 처리합니다. 직접 하면 id 리스트를 for문으로 만들고, collection 생성/관리도 수동입니다.

---

## 4. 유사도 검색 (Similarity Search)

| LangChain | 순수 Python |
|-----------|-------------|
| `database.similarity_search(query, k=3)` | `tax_collection.query(query_texts=query)` |

이 부분은 사실 비슷합니다. 둘 다 한 줄이고, ChromaDB가 직접 처리합니다.

---

## 5. LLM 질의

| LangChain | 순수 Python |
|-----------|-------------|
| `RetrievalQA.from_chain_type()` 체인 구성 → `qa_chain({"query": query})` | `ollama.chat()`으로 직접 system prompt에 검색 결과를 넣어서 호출 |

이게 가장 큰 차이입니다:

- **LangChain**: 검색 → 프롬프트 조합 → LLM 호출을 `RetrievalQA` 체인이 자동으로 연결해줍니다. Hub에서 검증된 프롬프트(`rlm/rag-prompt`)도 가져다 쓸 수 있습니다.
- **순수 Python**: 검색 결과를 직접 f-string으로 system prompt에 붙여서 LLM에 넘깁니다. 프롬프트도 직접 작성해야 합니다.

---

## 핵심 정리

**LangChain의 장점**: 각 단계가 추상화되어 있어서 코드가 짧고, 컴포넌트 교체(예: Ollama → OpenAI)가 쉽고, 체인으로 파이프라인이 자동 연결됨

**직접 구현의 장점**: 각 단계에서 정확히 무슨 일이 일어나는지 이해할 수 있고, 디버깅이 쉽고, 불필요한 의존성이 없음

> 결국 LangChain이 해주는 일은 "이 5단계를 편하게 연결해주는 것"이고, 직접 해보면 각 단계의 원리를 확실히 이해할 수 있다는 게 이 실습의 포인트입니다.

# LangChain RAG vs Pure Python RAG Comparison

A comparison based on the 5 stages of RAG.

---

## 1. Read Document

| LangChain | Pure Python |
|-----------|-------------|
| Single line: `Docx2txtLoader("./tax.docx")` | Create a Document object with `python-docx`, loop through paragraphs, and manually concatenate into `full_text` |

LangChain provides a unified `loader.load()` interface for various file formats (docx, pdf, csv, etc.), while doing it manually requires using separate libraries (`python-docx`, `PyPDF2`, etc.) to parse each format.

---

## 2. Split Text

| LangChain | Pure Python |
|-----------|-------------|
| `RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)` + `load_and_split()` | Manually write a function that encodes with `tiktoken` → slices → decodes |

LangChain natively supports `chunk_overlap` to prevent loss of context between chunks. The manual implementation simply slices tokens without overlap, which can cut sentences in the middle.

---

## 3. Embedding & Vector DB Storage

| LangChain | Pure Python |
|-----------|-------------|
| `OllamaEmbeddings()` + `Chroma.from_documents()` handles embedding and storage in one step | Manually create `chromadb.Client()`, create a collection, build an id list, and call `collection.add()` |

LangChain handles everything automatically when you pass in Document objects. Doing it manually requires building id lists with loops and managing collection creation yourself.

---

## 4. Similarity Search

| LangChain | Pure Python |
|-----------|-------------|
| `database.similarity_search(query, k=3)` | `tax_collection.query(query_texts=query)` |

This part is essentially the same. Both are one-liners, and ChromaDB handles the heavy lifting directly.

---

## 5. LLM Query

| LangChain | Pure Python |
|-----------|-------------|
| Build a chain with `RetrievalQA.from_chain_type()` → `qa_chain({"query": query})` | Call `ollama.chat()` directly, injecting search results into the system prompt |

This is the biggest difference:

- **LangChain**: The `RetrievalQA` chain automatically connects retrieval → prompt assembly → LLM call. You can also pull verified prompts from the Hub (e.g., `rlm/rag-prompt`).
- **Pure Python**: You manually insert search results into the system prompt using f-strings and call the LLM directly. Prompts must be written from scratch.

---

## Key Takeaways

**LangChain Advantages**: Each stage is abstracted, resulting in shorter code. Swapping components (e.g., Ollama → OpenAI) is easy, and chains automatically connect the pipeline.

**Manual Implementation Advantages**: You understand exactly what happens at each stage. Debugging is easier, and there are no unnecessary dependencies.

> Ultimately, what LangChain does is "conveniently connect these 5 stages." By implementing it manually, you gain a solid understanding of the underlying principles at each step — and that's the whole point of this exercise.